In [1]:
import pandas as pd
import numpy as np
import re

complaints = pd.read_csv('311_small.csv')

columns_to_drop = ['Created Date', 'Closed Date', 'Agency', 'Agency Name',
       'Descriptor', 'Status', 'Due Date',
       'Resolution Description', 'Resolution Action Updated Date',
       'Community Board', 'BBL', 'Open Data Channel Type',
       'Park Facility Name', 'Park Borough', 'Vehicle Type',
       'Taxi Company Borough', 'Taxi Pick Up Location', 'Bridge Highway Name',
       'Bridge Highway Direction', 'Road Ramp', 'Bridge Highway Segment']
df = complaints.drop(columns=columns_to_drop)

print(f"Shape: {df.shape}")
print(f"Type Zip: {df['Incident Zip'].dtype}")
print(f"np.nan value count: {df['Incident Zip'].isna().sum()}")

/var/folders/b3/cjvms6c10k77tgv6k_rw3jbh0000gn/T/ipykernel_76764/595314957.py:5: DtypeWarning: Columns (8,31,32,34,35,36,37) have mixed types. Specify dtype option on import or set low_memory=False.
  complaints = pd.read_csv('311_small.csv')


Shape: (999999, 20)
Type Zip: object
np.nan value count: 88509


Let's reduce this number.

In [2]:
all_zips = df['Incident Zip']

In [3]:
type(all_zips)

pandas.core.series.Series

In [4]:
def float_and_string(x):
    # make sure that we will have ONLY floats and string
    try:
        return float(x)
    except:
        return str(x)

In [5]:
zip1 = all_zips.apply(lambda x: float_and_string(x))

In [7]:
zip1.apply(type).unique()

array([<class 'float'>, <class 'str'>], dtype=object)

In [8]:
zip1[zip1.apply(lambda x: isinstance(x, str))]

883       37214-0065
40092         NEWARK
40145     10423-0935
63485     90054-0807
64033     18773-9640
76409     08690-1717
90617     11797-9004
146452    44087-2340
154503           UNK
159120             ?
164769    91716-0500
166800       NO IDEA
179235    30353-0942
179493             ?
209339       UNKNOWN
244742    30092-2670
253124             *
329115             *
379814       UNKNOWN
392099    53566-8019
419089    12551-0831
505758    17108-0988
508559    17108-0988
511704       UNKNOWN
549824    85251-3643
606653    11590-5027
654119    11735-3946
682171    11802-9060
685967             *
688809    61702-3517
760548    55164-0437
924175    61702-3517
957902       UNKNOWN
983209    14225-1032
Name: Incident Zip, dtype: object

In [9]:
import re
def clean_string(x):
    p1 = re.compile(r'[A-Z]+')
    p2 = re.compile(r'[?]+')
    p3 = re.compile(r'[*]+')
    if isinstance(x, str):
        for p in [p1, p2, p3]:
            if re.search(p, x):
                return np.nan
    return x

In [10]:
zip2 = zip1.apply(lambda x: clean_string(x))

In [11]:
zip2[zip2.apply(lambda x: isinstance(x, str))]

883       37214-0065
40145     10423-0935
63485     90054-0807
64033     18773-9640
76409     08690-1717
90617     11797-9004
146452    44087-2340
164769    91716-0500
179235    30353-0942
244742    30092-2670
392099    53566-8019
419089    12551-0831
505758    17108-0988
508559    17108-0988
549824    85251-3643
606653    11590-5027
654119    11735-3946
682171    11802-9060
688809    61702-3517
760548    55164-0437
924175    61702-3517
983209    14225-1032
Name: Incident Zip, dtype: object

In [14]:
def split_on_dash(x):
    p1 = re.compile(r'\d{5}')
    try:
        if re.match(p1, x):
            return float(x.split('-')[0])
    except TypeError:  # it's a float
        return x
        

In [15]:
zip3 = zip2.apply(lambda x: split_on_dash(x))

In [16]:
zip3[zip3.apply(lambda x: isinstance(x, str))]

Series([], Name: Incident Zip, dtype: float64)

In [17]:
zip3.isna().sum()

88521

In [19]:
complaints.iloc[146452, 16]

'TWINSBURG'

  * Manhattan: 10001-10282
  * Staten Island: 10301-10314
  * Bronx: 10451-10475
  * Queens: 11004-11109; 11351-11697
  * Brooklyn: 11201-11256

< 10001 
10282-10301 
10314-10451
10475-11004
11109-11201
11256-11351
> 11697

In [20]:
def clean_zip_ranges(x):
    try:
        code = int(x)
    except:
        return np.nan
    if code < 10001 or code > 11697:
        return np.nan
    elif 10282 < code < 10301:
        return np.nan
    elif 10314 < code < 10451:
        return np.nan
    elif 10475 < code < 11004:
        return np.nan
    elif 11109 < code < 11201:
        return np.nan
    elif 11256 < code < 11351:
        return np.nan
    else:
        return code

In [21]:
zip4 = zip3.apply(lambda x: clean_zip_ranges(x))

In [22]:
zip4.shape

(999999,)

In [23]:
zip4.isna().sum()

89526

In [24]:
zip4.dtype

dtype('float64')

In [25]:
df['Incident Zip'] = zip4